In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Efektivní simulace stabilizátorových Circuit s primitiivami Qiskit Aer

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
Tato stránka ukazuje, jak používat primitiva Qiskit Aer k efektivní simulaci stabilizátorových Circuit, včetně těch, které podléhají Pauliho šumu.

Stabilizátorové Circuit, známé také jako Cliffordovy Circuit, jsou důležitou omezenou třídou kvantových Circuit, které lze efektivně simulovat klasicky. Existuje několik ekvivalentních způsobů, jak definovat stabilizátorové Circuit. Jedna definice říká, že stabilizátorový Circuit je kvantový Circuit, který se skládá výhradně z následujících Gate:

- [CX](../api/qiskit/qiskit.circuit.library.CXGate)
- [Hadamard](../api/qiskit/qiskit.circuit.library.HGate)
- [S](../api/qiskit/qiskit.circuit.library.SGate)
- [Měření](../api/qiskit/circuit#qiskit.circuit.Measure)

Pomocí Hadamarda a S lze sestavit libovolný Gate pro Pauliho rotaci ([$R_x$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXGate), [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate) a [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate)) s úhlem z množiny ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$ (až na globální fázi), takže tyto Gate lze do definice zahrnout také.

Stabilizátorové Circuit jsou důležité pro studium kvantové korekce chyb. Jejich klasická simulovatelnost je také činí užitečnými pro ověřování výstupu kvantových počítačů. Představ si například, že chceš spustit kvantový Circuit využívající 100 Qubitů na kvantovém počítači. Jak zjistíš, že se kvantový počítač chová správně? Kvantový Circuit na 100 Qubitech je mimo dosah hrubé klasické simulace. Úpravou Circuit tak, aby se stal stabilizátorovým Circuit, můžeš na kvantovém počítači spouštět Circuit s podobnou strukturou jako tvůj požadovaný Circuit, ale které lze simulovat na klasickém počítači. Ověřením výstupu kvantového počítače na stabilizátorových Circuit získáš jistotu, že se správně chová i na nestabilizátorových Circuit. Příklad tohoto přístupu v praxi viz [*Evidence for the utility of quantum computing before fault tolerance*](https://www.nature.com/articles/s41586-023-06096-3).

[Přesná a zašuměná simulace s primitiivami Qiskit Aer](simulate-with-qiskit-aer) ukazuje, jak pomocí [Qiskit Aer](https://qiskit.org/ecosystem/aer/) provádět přesné a zašuměné simulace obecných kvantových Circuit. Uvažuj ukázkový Circuit z tohoto článku – Circuit s 8 Qubity sestavený pomocí [efficient_su2](../api/qiskit/qiskit.circuit.library.efficient_su2):

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg)

S Qiskit Aer jsme tento Circuit dokázali snadno simulovat. Předpokládejme ale, že nastavíme počet Qubitů na 500:

In [2]:
n_qubits = 500
circuit = efficient_su2(n_qubits)
# don't try to draw the circuit because it's too large

Protože náklady na simulaci kvantových Circuit rostou exponenciálně s počtem Qubitů, takový velký Circuit by obecně překročil schopnosti i výkonného simulátoru, jako je Qiskit Aer. Klasická simulace obecných kvantových Circuit se stává neproveditelnou, když počet Qubitů překročí přibližně 50 až 100. Všimni si však, že Circuit efficient_su2 je parametrizován úhly na Gate $R_y$ a $R_z$. Pokud jsou všechny tyto úhly z množiny ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$, pak je Circuit stabilizátorový a lze jej efektivně simulovat!

V následující buňce spustíme Circuit s primitivem Sampler podpořeným simulátorem stabilizátorových Circuit, přičemž parametry jsou zvoleny náhodně tak, aby byl Circuit zaručeně stabilizátorový.

In [3]:
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Initialize a Sampler backed by the stabilizer circuit simulator
exact_sampler = Sampler(
    options=dict(backend_options=dict(method="stabilizer"))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(
    1, AerSimulator(method="stabilizer")
)
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params)
job = exact_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Simulátor stabilizátorových Circuit také podporuje zašuměnou simulaci, ale pouze pro omezenou třídu šumových modelů. Konkrétně jakýkoli kvantový šum musí být charakterizován kanálem [Pauliho chyby](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.pauli_error.html#qiskit_aer.noise.pauli_error). [Depolarizační chyba](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.depolarizing_error.html) do této kategorie spadá, takže ji lze simulovat také. Simulovat lze i klasické šumové kanály jako [chyba čtení](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.ReadoutError.html).

Následující buňka kódu spouští stejnou simulaci jako dříve, tentokrát ale s šumovým modelem, který přidává depolarizační chybu 2 % ke každému Gate CX, a zároveň chybu čtení, která každý naměřený bit překlopí s pravděpodobností 5 %.

In [4]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
bit_flip_prob = 0.05
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)
noise_model.add_all_qubit_readout_error(
    ReadoutError(
        [
            [1 - bit_flip_prob, bit_flip_prob],
            [bit_flip_prob, 1 - bit_flip_prob],
        ]
    )
)

noisy_sampler = Sampler(
    options=dict(
        backend_options=dict(method="stabilizer", noise_model=noise_model)
    )
)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Nyní použijeme primitiv Estimator podpořený simulátorem stabilizátorových Circuit k výpočtu střední hodnoty observablu $ZZ \cdots Z$. Díky speciální struktuře stabilizátorových Circuit bude výsledek s velkou pravděpodobností 0.

In [5]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)

exact_estimator = Estimator(
    options=dict(backend_options=dict(method="stabilizer")),
)
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.0

## Next steps

<Admonition type="tip" title="Recommendations">
  - To simulate circuits with Qiskit Aer, see [Exact and noisy simulation with Qiskit Aer primitives](./simulate-with-qiskit-sdk-primitives).
  - Review the [Qiskit Aer](https://qiskit.org/ecosystem/aer/) documentation.
</Admonition>